# 10 - Extract → Conflict Detection

這是 agentic RAG 的核心能力之一：當同一個實體（客戶、決策、時程）
在多份文件中有不同描述時，自動偵測矛盾。

本節整合前面兩個筆記本的技術：
- **06 的 structured output**：用 Pydantic schema 萃取兩份文件的決策
- **09 的 markdown parser**：取得文件的基礎 metadata

並建立一個三節點 LangGraph：

```
START → [extract] → [compare] → [report] → END
```

| 節點 | 工作 |
|------|------|
| `extract` | 對每份文件呼叫 `with_structured_output()` 萃取決策與風險 |
| `compare` | 用相同的 `id` 配對兩份文件的條目，找出不同值 |
| `report`  | 格式化衝突清單，LLM 補充說明影響 |

In [1]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Annotated, TypedDict

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / 'examples'
CRM_DIR      = PROJECT_ROOT / 'data' / 'crm_notes'

if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from utils.tools import FileTools

load_dotenv(PROJECT_ROOT / '.env')
_crm = FileTools(CRM_DIR)

llm = ChatOpenAI(
    base_url=os.environ['VLLM_BASE_URL'],
    api_key=os.environ['VLLM_API_KEY'],
    model=os.environ['VLLM_MODEL'],
    temperature=0,
)
print(f'Project root: {PROJECT_ROOT}')
print(f'LLM: {os.environ["VLLM_MODEL"]}')

Project root: /home/mistin/research/agentic-rag-lab
LLM: gemma-4-31B-it


## 1 · Schema 定義

重用 06 的 `Decision` 與 `Risk` schema，並加上 `MeetingFacts`
作為單份文件的完整萃取結果。

In [2]:
class Decision(BaseModel):
    id: str = Field(description='決策編號，如 D-001')
    content: str = Field(description='決策內容摘要（30 字以內）')
    owner: str = Field(description='決策人')
    date: str = Field(description='決策日期 YYYY-MM-DD')


class Risk(BaseModel):
    id: str = Field(description='風險編號，如 R-001')
    description: str = Field(description='風險描述（30 字以內）')
    severity: str = Field(description='嚴重度：高/中/低')
    probability: str = Field(description='機率：高/中/低')


class MeetingFacts(BaseModel):
    customer: str
    meeting_date: str
    decisions: list[Decision]
    risks: list[Risk]

## 2 · Graph State

用 `TypedDict` 定義三個節點之間傳遞的狀態：
- `docs`：要比較的文件名稱清單
- `facts`：每份文件萃取出的 `MeetingFacts`
- `conflicts`：偵測到的衝突清單
- `report`：最終報告文字

In [3]:
@dataclass
class Conflict:
    field: str      # 衝突欄位（如 'decisions[D-001].content'）
    doc_a: str      # 第一份文件的值
    doc_b: str      # 第二份文件的值
    doc_a_name: str
    doc_b_name: str


class ConflictState(TypedDict):
    docs    : list[str]         # 要比較的文件檔名
    facts   : list[MeetingFacts]
    conflicts: list[Conflict]
    report  : str

## 3 · 三個節點

### 3.1 Extract Node

對 `state['docs']` 中的每份文件呼叫 `with_structured_output(MeetingFacts)`。

In [4]:
extractor = llm.with_structured_output(MeetingFacts)


def extract_node(state: ConflictState) -> dict:
    facts = []
    for doc_name in state['docs']:
        text = _crm.read_file(doc_name)
        result: MeetingFacts = extractor.invoke(
            f'請從以下會議紀錄萃取所有決策與風險項目：\n\n{text}'
        )
        facts.append(result)
        print(f'  [extract] {doc_name}: {len(result.decisions)} 決策, {len(result.risks)} 風險')
    return {'facts': facts}

### 3.2 Compare Node

比對兩份文件中相同 ID 的條目，找出值不同的欄位。
比對邏輯：
1. 建立 `{id → Decision/Risk}` 字典
2. 找出共同的 ID
3. 對每個共同 ID，比對 `content/description`、`severity` 等欄位

In [5]:
def compare_node(state: ConflictState) -> dict:
    if len(state['facts']) < 2:
        return {'conflicts': []}

    facts_a, facts_b = state['facts'][0], state['facts'][1]
    doc_a_name, doc_b_name = state['docs'][0], state['docs'][1]
    conflicts: list[Conflict] = []

    # 比對決策
    d_a = {d.id: d for d in facts_a.decisions}
    d_b = {d.id: d for d in facts_b.decisions}
    for did in set(d_a) & set(d_b):
        da, db = d_a[did], d_b[did]
        if da.content.strip() != db.content.strip():
            conflicts.append(Conflict(
                field=f'decisions[{did}].content',
                doc_a=da.content,
                doc_b=db.content,
                doc_a_name=doc_a_name,
                doc_b_name=doc_b_name,
            ))

    # 比對風險（同一 ID 的嚴重度是否有變化）
    r_a = {r.id: r for r in facts_a.risks}
    r_b = {r.id: r for r in facts_b.risks}
    for rid in set(r_a) & set(r_b):
        ra, rb = r_a[rid], r_b[rid]
        if ra.severity != rb.severity or ra.probability != rb.probability:
            field = f'risks[{rid}].severity/probability'
            val_a = f'嚴重度={ra.severity}, 機率={ra.probability}'
            val_b = f'嚴重度={rb.severity}, 機率={rb.probability}'
            conflicts.append(Conflict(
                field=field,
                doc_a=val_a,
                doc_b=val_b,
                doc_a_name=doc_a_name,
                doc_b_name=doc_b_name,
            ))

    print(f'  [compare] 發現 {len(conflicts)} 個衝突')
    return {'conflicts': conflicts}

### 3.3 Report Node

格式化衝突清單，並用 LLM 補充一句「影響說明」。

In [6]:
def report_node(state: ConflictState) -> dict:
    conflicts = state['conflicts']
    if not conflicts:
        return {'report': '未偵測到衝突。'}

    lines = [
        f'=== 衝突偵測報告 ===',
        f'比較文件：',
        f'  A: {state["docs"][0]}',
        f'  B: {state["docs"][1]}',
        f'發現衝突數：{len(conflicts)}',
        '',
    ]

    for i, c in enumerate(conflicts, start=1):
        lines.append(f'衝突 {i}：{c.field}')
        lines.append(f'  文件 A ({c.doc_a_name}): {c.doc_a}')
        lines.append(f'  文件 B ({c.doc_b_name}): {c.doc_b}')

        # LLM 補充影響說明
        impact_prompt = (
            f'以下是兩份會議紀錄之間的一個衝突：\n'
            f'欄位：{c.field}\n'
            f'舊值（{c.doc_a_name}）：{c.doc_a}\n'
            f'新值（{c.doc_b_name}）：{c.doc_b}\n\n'
            '請用一句話說明這個變更對專案的影響。'
        )
        impact = llm.invoke(impact_prompt).content.strip()
        lines.append(f'  影響：{impact}')
        lines.append('')

    report = '\n'.join(lines)
    return {'report': report}

## 4 · 組裝 Graph

In [7]:
_builder = StateGraph(ConflictState)
_builder.add_node('extract', extract_node)
_builder.add_node('compare', compare_node)
_builder.add_node('report',  report_node)
_builder.add_edge(START,     'extract')
_builder.add_edge('extract', 'compare')
_builder.add_edge('compare', 'report')
_builder.add_edge('report',  END)
conflict_detector = _builder.compile()
print('ConflictDetector graph 建立完成 ✓')

ConflictDetector graph 建立完成 ✓


## 5 · 執行：晨星科技 meeting_001 vs meeting_003

這兩份文件記錄了同一客戶的兩次會議，其中有已知的衝突：
- D-001：公有雲 → 私有雲
- R-003：嚴重度從「中」升為「高」

In [8]:
result = conflict_detector.invoke({
    'docs': [
        'meeting_001_晨星科技_2026-05-08.md',
        'meeting_003_晨星科技_2026-05-22.md',
    ],
    'facts': [],
    'conflicts': [],
    'report': '',
})

print()
print(result['report'])

  [extract] meeting_001_晨星科技_2026-05-08.md: 4 決策, 4 風險
  [extract] meeting_003_晨星科技_2026-05-22.md: 5 決策, 7 風險
  [compare] 發現 5 個衝突

=== 衝突偵測報告 ===
比較文件：
  A: meeting_001_晨星科技_2026-05-08.md
  B: meeting_003_晨星科技_2026-05-22.md
發現衝突數：5

衝突 1：decisions[D-001].content
  文件 A (meeting_001_晨星科技_2026-05-08.md): 採用方案 A（公有雲部署，AWS 台灣區），以 SaaS 模式導入
  文件 B (meeting_003_晨星科技_2026-05-22.md): 部署方案變更：放棄公有雲（方案 A），改採私有雲部署（方案 B，客戶自建機房）。
  影響：部署方案由公有雲（SaaS 模式）變更為私有雲（客戶自建機房），將導致部署架構、維運責任及導入模式的全面調整。

衝突 2：decisions[D-002].content
  文件 A (meeting_001_晨星科技_2026-05-08.md): 第一階段範疇鎖定財務模組，人資與供應鏈模組列為第二階段
  文件 B (meeting_003_晨星科技_2026-05-22.md): 第一階段範疇維持財務模組不變。
  影響：此次變更確認維持原定計畫，第一階段範疇仍鎖定於財務模組，對專案進度與範圍無實質影響。

衝突 3：decisions[D-004].content
  文件 A (meeting_001_晨星科技_2026-05-08.md): 技術規格書由我方（陳建宏）撰寫，客戶方（張志偉）確認
  文件 B (meeting_003_晨星科技_2026-05-22.md): 技術規格書需補充「私有雲架構圖」，由陳建宏於 2026-05-27 提交修訂版。
  影響：此變更將原有的撰寫分工明確化，並新增了「私有雲架構圖」的交付要求及 2026-05-27 的提交期限。

衝突 4：decisions[D-003].content
  文件 A (meeting_001_晨星科技_2026-05-08.md): 目標上

## 6 · 擴展：自動掃描同一客戶的所有文件

對同一客戶有超過兩份會議紀錄時，可以依時間排序後，
對相鄰兩份文件執行衝突偵測，追蹤決策演變。

In [9]:
import re as _re


def get_docs_for_customer(customer_keyword: str) -> list[str]:
    """回傳指定客戶的所有文件，依日期排序。"""
    all_files = _crm.list_files()
    matched = [f for f in all_files if customer_keyword in f]
    # 從檔名萃取日期排序
    def extract_date(fname: str) -> str:
        m = _re.search(r'(\d{4}-\d{2}-\d{2})', fname)
        return m.group(1) if m else ''
    return sorted(matched, key=extract_date)


docs = get_docs_for_customer('晨星科技')
print('晨星科技 文件清單（依日期排序）：')
for d in docs:
    date = _re.search(r'(\d{4}-\d{2}-\d{2})', d)
    print(f'  {date.group(1) if date else "??"}  {d}')

print()
# 掃描相鄰文件對
for i in range(len(docs) - 1):
    doc_pair = [docs[i], docs[i + 1]]
    print(f'掃描 {Path(doc_pair[0]).stem.split("_")[1]} vs {Path(doc_pair[1]).stem.split("_")[1]} ... ', end='')
    r = conflict_detector.invoke({
        'docs': doc_pair, 'facts': [], 'conflicts': [], 'report': '',
    })
    n = len(r['conflicts'])
    print(f'→ {n} 個衝突')

晨星科技 文件清單（依日期排序）：
  2026-05-08  meeting_001_晨星科技_2026-05-08.md
  2026-05-22  meeting_003_晨星科技_2026-05-22.md
  2026-06-05  meeting_005_晨星科技_2026-06-05.md

掃描 001 vs 003 ...   [extract] meeting_001_晨星科技_2026-05-08.md: 4 決策, 4 風險
  [extract] meeting_003_晨星科技_2026-05-22.md: 5 決策, 7 風險
  [compare] 發現 8 個衝突
→ 8 個衝突
掃描 003 vs 005 ...   [extract] meeting_003_晨星科技_2026-05-22.md: 5 決策, 7 風險
  [extract] meeting_005_晨星科技_2026-06-05.md: 2 決策, 4 風險
  [compare] 發現 2 個衝突
→ 2 個衝突


## 小結

Extract → Conflict Detection 的架構整合了本 lab 的多個模組：

```
FileTools.read_file()           ← 01：檔案存取
  + with_structured_output()   ← 06：結構化萃取
  + LangGraph StateGraph       ← 02：圖形化 agent
  → 衝突偵測報告               ← 本節：多文件推理
```

**關鍵設計決策**：
- 比對以 `id`（D-001、R-003）為 key，而非語意相似度，確保確定性
- 衝突判斷是純 Python（字串比較），LLM 只負責「說明影響」
- 這讓衝突偵測本身可測試、可重現，不依賴 LLM 的判斷能力

---
**後續延伸方向**：
- 加入 `pending_decisions` 狀態：某些決策在 A 文件存在但 B 文件不見了（可能被取消）
- 整合 09 的 JSONL chunks，對所有 sections 進行衝突掃描而不只是決策/風險
- 加入 Vespa 向量搜尋（Phase 3），讓 extract 階段可以先 retrieve 再 extract

## 7 · 文字型衝突偵測（LLM 語意比對）

### ID-key 方法的盲點

前面的 `compare_node` 依賴**相同的 ID**（D-001、R-003）做配對。
但當新文件沒有沿用舊 ID、而是在敘述文字中隱含矛盾時，ID-key 方法就會靜默地漏掉衝突：

```
meeting_003  決策 D-003 → 目標上線日期：2026-06-15（在表格）
meeting_005  決策 D-005 → 追加驗證（不同 ID，沒有 D-003）
             補充備註   → 「最快 7 月上旬才能確認上線」（在純文字）
```

`compare_node` 看不到上面的矛盾，因為 meeting_005 根本沒有 D-003 可以比對。

**文字型偵測**的做法：直接把兩份文件的敘述性段落送給 LLM，
讓它判斷「這兩段話是否有相互矛盾之處」。

In [10]:
# 先用 ID-key 方法跑 meeting_003 vs meeting_005，看它漏掉了什麼
result_miss = conflict_detector.invoke({
    'docs': [
        'meeting_003_晨星科技_2026-05-22.md',
        'meeting_005_晨星科技_2026-06-05.md',
    ],
    'facts': [],
    'conflicts': [],
    'report': '',
})

print('=== ID-key 比對結果（meeting_003 vs meeting_005）===')
print(result_miss['report'])
print()
print('⚠️  注意：Go-Live 日期（6/15 vs 7月上旬）和遷移驗證次數（1次 vs 追加）的矛盾都沒有被偵測到。')


  [extract] meeting_003_晨星科技_2026-05-22.md: 5 決策, 7 風險
  [extract] meeting_005_晨星科技_2026-06-05.md: 2 決策, 3 風險
  [compare] 發現 2 個衝突
=== ID-key 比對結果（meeting_003 vs meeting_005）===
=== 衝突偵測報告 ===
比較文件：
  A: meeting_003_晨星科技_2026-05-22.md
  B: meeting_005_晨星科技_2026-06-05.md
發現衝突數：2

衝突 1：decisions[D-005].content
  文件 A (meeting_003_晨星科技_2026-05-22.md): 客戶（張志偉）需於 2026-05-25 前確認私有雲硬體規格並啟動採購流程。
  文件 B (meeting_005_晨星科技_2026-06-05.md): 追加第二輪資料遷移沙盒驗證（預計 2026-06-12 執行）。
  影響：此變更將原定的硬體規格確認與採購時程，調整為追加第二輪資料遷移沙盒驗證，可能導致後續硬體採購與部署進度延後。

衝突 2：risks[R-005].severity/probability
  文件 A (meeting_003_晨星科技_2026-05-22.md): 嚴重度=高, 機率=高
  文件 B (meeting_005_晨星科技_2026-06-05.md): 嚴重度=中, 機率=高
  影響：風險 R-005 的嚴重度從「高」調降至「中」，顯示該風險對專案的潛在威脅程度有所降低。


⚠️  注意：Go-Live 日期（6/15 vs 7月上旬）和遷移驗證次數（1次 vs 追加）的矛盾都沒有被偵測到。


### 文字型偵測實作

`TextContradiction` schema 要求 LLM 輸出：
- `summary`：矛盾描述
- `quote_a` / `quote_b`：各自的原文摘錄（作為證據）
- `conflict_type`：矛盾分類

比對的單位是**段落（section）**，而非整份文件，
避免輸入過長導致 LLM 遺漏細節。

In [11]:
import re as _re2


# ── Schema ────────────────────────────────────────────────────────────
class TextContradiction(BaseModel):
    summary: str = Field(description='矛盾摘要（一句話）')
    quote_a: str = Field(description='文件 A 中的相關原文（盡量引用原句）')
    quote_b: str = Field(description='文件 B 中的相關原文（盡量引用原句）')
    conflict_type: str = Field(description='矛盾類型：時程衝突 / 決策反轉 / 數字矛盾 / 其他')


class TextContradictions(BaseModel):
    contradictions: list[TextContradiction]


# ── 輔助：從 .md 文字中取出指定 section ───────────────────────────────
def extract_section(md_text: str, section_title: str) -> str:
    """取出指定 ## 標題下的內容，找不到時回傳空字串。"""
    pattern = rf'^## {_re2.escape(section_title)}\s*\n(.*?)(?=^## |\Z)'
    m = _re2.search(pattern, md_text, _re2.MULTILINE | _re2.DOTALL)
    return m.group(1).strip() if m else ''


# ── 核心函式 ──────────────────────────────────────────────────────────
text_conflict_detector = llm.with_structured_output(TextContradictions)


def detect_text_conflicts(
    text_a: str,
    text_b: str,
    doc_a_name: str,
    doc_b_name: str,
) -> TextContradictions:
    """
    給定兩段文字，請 LLM 找出語意層面的矛盾。
    兩段文字可以來自不同 section、不同格式（表格、條列、散文均可）。
    """
    prompt = (
        f'請比較以下來自兩份會議紀錄的文字段落，找出所有相互矛盾的陳述。\n'
        f'矛盾包括：同一事實有不同描述、數字不一致、決策被推翻、時程前後衝突等。\n\n'
        f'【文件 A】{doc_a_name}\n'
        f'{text_a}\n\n'
        f'【文件 B】{doc_b_name}\n'
        f'{text_b}\n\n'
        f'請找出所有矛盾，若確實沒有矛盾，回傳空的 contradictions 清單。'
    )
    return text_conflict_detector.invoke(prompt)


### 執行：比對 meeting_003 與 meeting_005 的敘述段落

選取兩份文件中**格式最自由、最不適合 ID-key 比對**的 sections：
- meeting_003：`決策紀錄`（含 D-003 上線日期）+ `補充備註`（含遷移風險私下說法）
- meeting_005：`進度摘要`（含追加驗證決定）+ `補充備註`（含 7 月上旬說法）

這四個段落混合了表格、條列和散文，ID-key 無法處理，但 LLM 可以跨格式理解。

In [12]:
DOC_A = 'meeting_003_晨星科技_2026-05-22.md'
DOC_B = 'meeting_005_晨星科技_2026-06-05.md'

text_003 = _crm.read_file(DOC_A)
text_005 = _crm.read_file(DOC_B)

# 取出要比對的段落：只送 LLM 最相關的 section，不送整份文件
section_a = '\n\n'.join([
    extract_section(text_003, '決策紀錄'),
    extract_section(text_003, '補充備註'),
])
section_b = '\n\n'.join([
    extract_section(text_005, '進度摘要'),
    extract_section(text_005, '補充備註'),
])

print(f'文件 A 段落長度: {len(section_a)} chars')
print(f'文件 B 段落長度: {len(section_b)} chars')


文件 A 段落長度: 803 chars
文件 B 段落長度: 717 chars


In [13]:
contradictions = detect_text_conflicts(section_a, section_b, DOC_A, DOC_B)

print(f'=== 文字型衝突偵測結果：{DOC_A} vs {DOC_B} ===')
print(f'發現矛盾數：{len(contradictions.contradictions)}')
print()

for i, c in enumerate(contradictions.contradictions, start=1):
    print(f'矛盾 {i}【{c.conflict_type}】')
    print(f'  摘要  : {c.summary}')
    print(f'  文件 A: {c.quote_a}')
    print(f'  文件 B: {c.quote_b}')
    print()


=== 文字型衝突偵測結果：meeting_003_晨星科技_2026-05-22.md vs meeting_005_晨星科技_2026-06-05.md ===
發現矛盾數：3

矛盾 1【時程衝突/決策落差】
  摘要  : 上線日期（Go-Live Date）不一致
  文件 A: 上線日期調整為 2026-06-15（原為 2026-07-01）...此決策不可撤回。
  文件 B: 張志偉私下表示，以目前進度評估，最快也要 7 月上旬才能確認上線。這與 2026-05-22 會議記錄中的「2026-06-15 Go-Live」計畫產生落差。

矛盾 2【執行計畫與風險評估衝突】
  摘要  : 資料遷移驗證次數之執行與評估衝突
  文件 A: 陳建宏私下表示：將遷移驗證從 3 次縮為 1 次風險很高
  文件 B: 陳建宏評估，94% 通過率不足以支撐直接上線，建議追加一輪補充驗證（原訂 1 次的計畫需重新評估）。

矛盾 3【時程邏輯衝突（API 釋出日與上線日相同，導致無法進行整合測試即上線）】
  摘要  : 銀行 API 整合進度與上線日期的邏輯矛盾
  文件 A: 上線日期調整為 2026-06-15
  文件 B: 國泰銀行 API 問題...預計需等待對方 2026-06-15 版本釋出後才能進行整合測試。



### 兩種方法對比

| 面向 | ID-key 比對 | LLM 文字比對 |
|------|-------------|--------------|
| **能偵測的矛盾** | 相同 ID 的欄位值不同 | 任何語意層面的矛盾 |
| **需要結構化資料** | 是（Pydantic schema） | 否（純文字輸入） |
| **確定性** | 高（純 Python 字串比較） | 低（依賴 LLM 判斷） |
| **可測試性** | 容易（輸出是 Conflict dataclass） | 較難（LLM 輸出可能每次不同） |
| **適合的衝突類型** | 表格型資料（決策、風險、時程） | 散文、跨 section 的隱性矛盾 |

**設計建議**：兩種方法互補，不要選一個捨棄另一個。
實務上可以先跑 ID-key（確定性高、快），再對 ID-key 沒有覆蓋的敘述性段落補跑 LLM 比對。